# Урок 20. IP-адресация и маски подсети

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-19.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-21.ipynb)

---

Структура IP-адреса. Маска подсети, адрес сети и адрес узла. Разбор классических задач ЕГЭ на маски. Проверка ответа кодом.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-20", name=ФИО, klass=Класс)

## Разбираемся

### IP-адрес — это 32 бита

Привычная запись `192.168.1.10` — просто удобная форма. На самом
деле это **32 бита**, разбитые на четыре байта по 8 бит:

```
192      168      1        10
11000000 10101000 00000001 00001010
```

Отсюда сразу два факта, которые спрашивают на ЕГЭ:

* каждое число в адресе — от 0 до 255 (восемь бит);
* всего адресов $2^{32}$ ≈ 4,3 миллиарда.

### Адрес сети и адрес узла

Адрес делится на две части: **какая сеть** и **какой узел в ней**.
Где именно проходит граница, задаёт **маска подсети**.

Маска — тоже 32 бита, но устроена жёстко: сначала все единицы,
потом все нули. Единицы отмечают часть сети, нули — часть узла.

```
адрес  192.168.1.10   11000000 10101000 00000001 00001010
маска  255.255.255.0  11111111 11111111 11111111 00000000
                      └──────── сеть ──────────┘└─ узел ─┘
```

### Главная формула

> **Адрес сети = адрес узла И маска** (побитовое «и»).

Побитовое «и» в Python — это `&`. Там, где в маске единица, бит
адреса сохраняется; где ноль — обнуляется.

```python
192 & 255 = 192
10  & 0   = 0
```

Значит, для нашего примера адрес сети — `192.168.1.0`.

### Сколько узлов в сети

Если в маске $k$ нулей, то под номер узла отведено $k$ бит,
то есть $2^k$ комбинаций. Но два адреса заняты всегда:

* **все нули** — это адрес самой сети;
* **все единицы** — широковещательный адрес.

> Узлов в сети: $2^k - 2$.

| Маска | Нулей | Всего | Узлов |
|---|---|---|---|
| 255.255.255.0 | 8 | 256 | 254 |
| 255.255.255.128 | 7 | 128 | 126 |
| 255.255.255.192 | 6 | 64 | 62 |
| 255.255.255.240 | 4 | 16 | 14 |
| 255.255.255.252 | 2 | 4 | 2 |

Эту табличку стоит уметь восстанавливать: маска 255.255.255.240 —
это 240 = 11110000, четыре нуля, значит 14 узлов.

### Короткая запись

`192.168.1.0/24` означает «маска из 24 единиц», то есть
255.255.255.0. Число после косой черты называют **префиксом**.

Префикс $p$ → нулей $32 - p$ → узлов $2^{32-p} - 2$.

### Типичные задачи ЕГЭ

1. **Дан адрес и маска — найти адрес сети.** Побитовое «и»
   по каждому байту.
2. **Дан адрес сети и маска — сколько узлов.** Считаем нули в маске.
3. **Даны два адреса — в одной ли они сети.** Сравниваем адреса
   сетей.
4. **Найти третий слева байт адреса сети.** Тот же расчёт,
   но в ответ идёт одно число.

Все четыре решаются одной и той же операцией — и все четыре
легко проверить кодом.

### Как считать вручную

Байты, где маска равна 255, переписываются без изменений.
Байты, где маска равна 0, обнуляются. Считать по битам приходится
только там, где маска — что-то промежуточное вроде 192 или 240.

## Смотрим, как это работает

### Пример 1. Адрес в битах

In [ ]:
адрес = "192.168.1.10"
байты = [int(ч) for ч in адрес.split(".")]

print(адрес)
print(" ".join(f"{б:08b}" for б in байты))
print("Всего бит:", len(байты) * 8)

### Пример 2. Адрес сети через побитовое «и»

In [ ]:
def адрес_сети(адрес, маска):
    a = [int(ч) for ч in адрес.split(".")]
    m = [int(ч) for ч in маска.split(".")]
    return ".".join(str(a[i] & m[i]) for i in range(4))


примеры = [
    ("192.168.1.10", "255.255.255.0"),
    ("192.168.1.200", "255.255.255.192"),
    ("10.11.12.13", "255.240.0.0"),
]

for адрес, маска in примеры:
    print(f"{адрес:<16} и {маска:<16} → {адрес_сети(адрес, маска)}")

### Пример 3. Что происходит по битам

In [ ]:
адрес, маска = 200, 192

print(f"адрес {адрес:>3} = {адрес:08b}")
print(f"маска {маска:>3} = {маска:08b}")
print(f"       и   = {адрес & маска:08b} = {адрес & маска}")

Видно механику: первые два бита маски — единицы, они сохраняют
биты адреса; остальные нули всё обнуляют.

### Пример 4. Сколько узлов

In [ ]:
def узлов(маска):
    биты = "".join(f"{int(ч):08b}" for ч in маска.split("."))
    нулей = биты.count("0")
    return 2 ** нулей - 2


for маска in ["255.255.255.0", "255.255.255.128", "255.255.255.192",
              "255.255.255.240", "255.255.0.0"]:
    биты = "".join(f"{int(ч):08b}" for ч in маска.split("."))
    print(f"{маска:<16} нулей {биты.count('0'):>2} → узлов {узлов(маска):>6,}")

### Пример 5. Префикс

In [ ]:
for префикс in [24, 25, 26, 28, 30, 16]:
    нулей = 32 - префикс
    маска_бит = "1" * префикс + "0" * нулей
    байты = [int(маска_бит[i:i + 8], 2) for i in range(0, 32, 8)]
    маска = ".".join(map(str, байты))
    print(f"/{префикс:<3} → {маска:<16} узлов {2 ** нулей - 2:>6,}")

### Пример 6. В одной ли сети

In [ ]:
пары = [
    ("192.168.1.10", "192.168.1.200", "255.255.255.0"),
    ("192.168.1.10", "192.168.1.200", "255.255.255.192"),
    ("10.0.0.1", "10.0.1.1", "255.255.0.0"),
]

for первый, второй, маска in пары:
    сеть1 = адрес_сети(первый, маска)
    сеть2 = адрес_сети(второй, маска)
    вердикт = "в одной сети" if сеть1 == сеть2 else "в разных сетях"
    print(f"{первый:<14} и {второй:<14} при {маска:<16} → {вердикт}")

Вторая строка показывает, зачем нужны нестандартные маски:
те же два адреса при более узкой маске оказываются в разных сетях.

### Пример 7. Задача ЕГЭ целиком

«Узел с адресом 111.81.27.87 находится в сети с маской
255.255.255.192. Каков третий слева байт адреса сети?»

In [ ]:
адрес = "111.81.27.87"
маска = "255.255.255.192"

сеть = адрес_сети(адрес, маска)
третий = сеть.split(".")[2]

print("Адрес сети:", сеть)
print("Третий слева байт:", третий)
print()
print("Проверка по битам последнего значащего байта:")
print(f"  87  = {87:08b}")
print(f"  192 = {192:08b}")
print(f"  и   = {87 & 192:08b} = {87 & 192}")

## Пробуем сами

### Задача 1. Сколько бит в IP-адресе

Сколько бит занимает адрес версии IPv4?

In [ ]:
#@title 🧩 Задача 1. Размер адреса { display-mode: "form" }
#@markdown Впиши число бит
бит = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", бит, "e29c9c180c6279b0",
         hint="Четыре байта.")

### Задача 2. Маска для префикса /24

Какой маске соответствует запись `/24`?

In [ ]:
#@title 🧩 Задача 2. Маска по префиксу { display-mode: "form" }
#@markdown Впиши маску в обычной записи
маска_24 = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", маска_24, "38da7009488a103f",
         hint="Двадцать четыре единицы — это три полных байта.")

### Задача 3. Адрес сети

Функция получает адрес и маску строками и возвращает адрес сети
строкой.

In [ ]:
def сеть(адрес, маска):
    return ...

In [ ]:
si.check("3", сеть, [
    (("192.168.1.10", "255.255.255.0"), "192.168.1.0"),
    (("192.168.1.200", "255.255.255.192"), "192.168.1.192"),
    (("111.81.27.87", "255.255.255.192"), "111.81.27.64"),
    (("10.11.12.13", "255.0.0.0"), "10.0.0.0"),
])

### Задача 4. Сколько узлов

Функция получает маску строкой и возвращает количество адресов,
которые можно выделить узлам: два адреса заняты всегда.

In [ ]:
def узлов_в_сети(маска):
    return ...

In [ ]:
si.check("4", узлов_в_сети, [
    ("255.255.255.0", 254),
    ("255.255.255.192", 62),
    ("255.255.255.240", 14),
    ("255.255.0.0", 65534),
])

### Задача 5. В одной ли сети

Функция получает два адреса и маску. Возвращает `True`, если узлы
находятся в одной сети.

In [ ]:
def одна_сеть(первый, второй, маска):
    return ...

In [ ]:
si.check("5", одна_сеть, [
    (("192.168.1.10", "192.168.1.200", "255.255.255.0"), True),
    (("192.168.1.10", "192.168.1.200", "255.255.255.192"), False),
    (("10.0.0.1", "10.0.1.1", "255.255.0.0"), True),
    (("10.0.0.1", "11.0.0.1", "255.0.0.0"), False),
])

### Задача 6. Маска по префиксу

Функция получает префикс (число от 0 до 32) и возвращает маску
в обычной записи.

In [ ]:
def маска_по_префиксу(префикс):
    return ...

In [ ]:
si.check("6", маска_по_префиксу, [
    (24, "255.255.255.0"),
    (26, "255.255.255.192"),
    (16, "255.255.0.0"),
    (32, "255.255.255.255"),
])

## Домашнее задание

### Домашнее задание 1. Разбор пяти задач ЕГЭ

Найдите в сборнике или на сайте ФИПИ пять задач на маски подсети
и решите каждую **дважды**: вручную по битам и своей функцией
из задачи 3.

Выпишите те, где ручной ответ разошёлся с машинным, и найдите,
где ошиблись. Покажите работу учителю.

### Домашнее задание 2. Третий байт адреса сети

Функция получает адрес и маску и возвращает **третий слева байт**
адреса сети числом. Именно в такой форме ответ просят на ЕГЭ.

In [ ]:
def третий_байт(адрес, маска):
    return ...

In [ ]:
si.check("дз2", третий_байт, [
    (("111.81.27.87", "255.255.255.192"), 27),
    (("192.168.200.10", "255.255.192.0"), 192),
    (("10.11.12.13", "255.255.0.0"), 0),
])

### Домашнее задание 3. Сколько адресов в байте

Сколько различных значений может принимать один байт IP-адреса?

In [ ]:
#@title 🏠 Домашнее 3. Значений в байте { display-mode: "form" }
#@markdown Впиши число
значений = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", значений, "51e8ea280b44e169",
         hint="От 0 до 255 включительно.")

---

### Любопытно

Четыре миллиарда адресов в 1981 году казались числом, которого
хватит навсегда: компьютеров в мире было несколько тысяч.
Свободные адреса закончились в 2011 году.

Спасает временная мера: дома и в школе устройства сидят
на «внутренних» адресах вроде 192.168.x.x, а наружу выходят через
один общий. Поэтому ваш телефон и ноутбук для всего интернета —
один адрес.

Настоящее решение — IPv6 со 128 битами: адресов там $2^{128}$,
это примерно по миллиарду миллиардов на каждый квадратный
миллиметр поверхности Земли. Переход идёт уже двадцать лет.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-19.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-21.ipynb)